# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, referencing all data entities by their `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("Dataset @id:", metadata['@id'])

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

To explore the record sets, fields, and columns, we retrieve them using the dataset object and print their `@id`s and descriptions.

In [ ]:
# List available record sets by their @id and name
record_sets = dataset.record_sets
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    print(f'  @id: {rs["@id"]}, name: {rs.get("name", "N/A")}, description: {rs.get("description", "")}')

# Select a record set for further exploration
if record_sets:
    main_record_set = record_sets[0]["@id"]
    print("\nMain record set selected:", main_record_set)
else:
    raise ValueError("No record sets defined in schema.")

# Review fields for that record set
fields = [f for f in dataset.fields if f['record_set'] == main_record_set]
print(f"\nFields for record set {main_record_set}:")
for field in fields:
    fid = field['@id']
    print(f'  Field @id: {fid}, name: {field.get("name", "N/A")}, type: {field.get("dataType", "N/A")}')

# Review columns for that record set
columns = [c for c in dataset.columns if c['record_set'] == main_record_set]
print(f"\nColumns for record set {main_record_set}:")
for col in columns:
    cid = col['@id']
    print(f'  Column @id: {cid}, name: {col.get("name", "N/A")}, source: {col.get("source", "N/A")}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We reference the record set and field `@id`s found above.

In [ ]:
# Extract data from all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records_list = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records_list)

# Display one record set's columns
print(f"Columns in DataFrame for record set {main_record_set}:")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering records, normalizing numeric fields, and grouping. Reference all fields by their `@id`.

We'll identify a numeric field (such as GAD-7 score, PHQ-9 score, or PSQ score) from the metadata and show typical analysis.

In [ ]:
# Identify a numeric field (by @id, for example: GAD-7 score)
gad7_field_id = None
# Find a GAD-7 field by name
for f in dataset.fields:
    if 'GAD-7' in f.get('name', ''):
        gad7_field_id = f['@id']
        break

print("GAD-7 Score Field @id:", gad7_field_id)

# If not found, fallback to a likely column name
gad7_field_id = gad7_field_id or 'gad_7_score'

df = dataframes[main_record_set]

if gad7_field_id not in df.columns:
    print(f"Field {gad7_field_id} not in DataFrame columns. Using available numeric column.")
    for col in df.columns:
        if 'score' in col.lower():
            gad7_field_id = col
            print(f"Fallback to column: {col}")
            break

# Filter rows with high GAD-7 score (>10)
threshold = 10
if gad7_field_id in df.columns:
    filtered_df = df[df[gad7_field_id] > threshold].copy()
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize GAD-7 score
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by a demographic field (e.g., gender, referenced by @id)
    group_field_id = None
    for f in dataset.fields:
        if 'gender' in f.get('name', '').lower():
            group_field_id = f['@id']
            break
    group_field_id = group_field_id or 'gender'
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        print(f"\nGrouped mean {gad7_field_id} by {group_field_id} for filtered records:")
        print(grouped_df)
else:
    print(f"Numeric field {gad7_field_id} not found for analysis.")

## 5. Visualization
Visualize data distributions and relationships between fields (referenced by their `@id`). Example: plot the GAD-7 score distribution and gender group means.

In [ ]:
# Plot GAD-7 score distribution
if gad7_field_id in df.columns:
    plt.figure(figsize=(9,5))
    sns.histplot(df[gad7_field_id].dropna(), bins=15, kde=True, color='steelblue')
    plt.title(f"Distribution of {gad7_field_id} (GAD-7 Score)")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Frequency")
    plt.show()

    # Visualize group mean if available
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        plt.figure(figsize=(7,4))
        sns.barplot(x=group_field_id, y=gad7_field_id, data=group_means, palette='viridis')
        plt.title(f"Mean {gad7_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {gad7_field_id}")
        plt.show()

## 6. Conclusion
Summarize findings from the data exploration:

- Loaded the FAIR² mental health dataset from Kilifi County using `mlcroissant`.
- Referenced all record sets, fields, and columns using their `@id`s for consistency.
- Identified numeric fields (e.g., GAD-7 score), filtered, normalized, and grouped them to illustrate EDA.
- Visualized data distributions (GAD-7 scores and group means).

This notebook demonstrates reproducible, standards-based processing of Croissant-defined datasets for AI-ready use cases.